# Machine Learning over Structured Data — Benchmark and Baselines

**Running example:** RelBench `rel-f1`, task `driver-position`
*For each driver who raced in the past year, predict their average finishing position over the next 60 days.*

Outline
1. Load a relational database and look at its tables, keys and time columns
2. Load a prediction task and its temporal train / val / test split
3. Baselines: global mean and **per-driver historical mean** (hands-on)
4. Flatten the database into one feature table (no leakage!)
5. Demo: TabPFN on the flat table
6. Compare results

In [ ]:
%pip install -q -U "relbench>=3.0"

## 1. The database

In [ ]:
import numpy as np
import pandas as pd
import relbench

dataset = relbench.load_dataset("rel-f1")
db = dataset.get_db()          # rows after the test timestamp are hidden
print("val timestamp: ", dataset.val_timestamp)
print("test timestamp:", dataset.test_timestamp)

In [ ]:
# Every table: size, primary key, foreign keys (column -> referenced table), time column
for name, table in db.table_dict.items():
    print(f"{name:22s} rows={len(table.df):6d}  pk={table.pkey_col!s:24s} "
          f"fks={table.fkey_col_to_pkey_table}  time={table.time_col}")

In [ ]:
drivers = db.table_dict["drivers"].df
results = db.table_dict["results"].df
races = db.table_dict["races"].df
results.head()

**Try it:** follow one row of `results` through its foreign keys. Which driver, which race, which circuit?

In [ ]:
row = results.iloc[0]
race = races.set_index("raceId").loc[row["raceId"]]
driver = drivers.set_index("driverId").loc[row["driverId"]]
print(driver["forename"], driver["surname"], "finished", row["positionOrder"], "in", race["name"], race["year"])

## 2. The task

In [ ]:
task = dataset.load_task("driver-position") # For each driver who raced in the past year, predict their average finishing position over the next 60 days.
print("type:", task.task_type, "| entity:", task.entity_col, "| target:", task.target_col,
      "| horizon:", task.timedelta)

train_df = task.get_table("train").df
val_df = task.get_table("val").df
test_df = task.get_table("test").df      # target column is hidden

for name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(f"{name:5s} {df.shape}  {df['date'].min().date()} -> {df['date'].max().date()}")
train_df.head()

Each row is *(driverId, date, position)*: "standing at `date`, what will this driver's average finishing position be over the next 60 days?"

The split is **temporal**: all train dates < val dates < test dates.

RelBench scores regression with **NMAE** = MAE / (std of the train target). We also print plain MAE (in finishing positions), which is easier to interpret.

In [ ]:
from sklearn.metrics import mean_absolute_error

val_table = task.get_table("val")
results_table = {}

def report(name, pred):
    """Evaluate predictions for the val rows and store the result."""
    pred = np.asarray(pred, dtype=float)
    nmae = task.evaluate(pred, val_table)["nmae"]
    mae = mean_absolute_error(val_df["position"], pred)
    results_table[name] = {"val MAE": mae, "val NMAE": nmae}
    print(f"{name:25s} MAE={mae:.3f}  NMAE={nmae:.3f}")

## 3. Baselines

### 3a. Global mean
Predict the same number for everyone: the mean of the training targets.

In [ ]:
global_mean = train_df["position"].mean()
pred_global = np.full(len(val_df), global_mean)
report("global mean", pred_global)

### 3b. Per-driver historical mean (hands-on)

$$\hat{y}_{d} = \frac{1}{|H_d|} \sum_{t \in H_d} y_{d,t}$$

where $H_d$ are driver $d$'s **training** rows. All of them are dated before the validation timestamp, so this uses no future information.

In [ ]:
# EXERCISE: per-driver historical mean
# 1. For every driver, compute the mean `position` over the TRAIN table only.
# 2. Map those means onto the validation rows (by driverId).
# 3. Drivers that never appear in train -> fall back to `global_mean`.

driver_means = train_df.groupby("driverId")["position"].mean()
pred_entity = (val_df["driverId"].map(driver_means).fillna(global_mean))

ev_nmae = task.evaluate(pred_entity, val_table)["nmae"]
task.evaluate(pred_entity, val_table)
# report("entity mean", pred_entity)

results_table = {
    "MNAE": ev_nmae,
    "MAE":mean_absolute_error(val_df["position"], pred_entity)
}

print(results_table)

In [ ]:
# STRETCH (optional): use only each driver's last k training observations.
# Does a "recent" mean beat the all-time mean? Why might it?

## 4. Flattening: one row per (driver, date)

TabPFN needs a single table. For each row of the task table we aggregate the driver's **past** results, meaning only races strictly before `date`. Using all of `results` would leak the future.

In [ ]:
import duckdb

FEATURE_SQL = """
SELECT
    t.driverId,
    t.date,
    COUNT(r.resultId)                                   AS n_races,
    AVG(r.positionOrder)                                AS mean_finish,
    AVG(r.grid)                                         AS mean_grid,
    SUM(r.points)                                       AS total_points,
    AVG(CASE WHEN r.date > t.date - INTERVAL 1 YEAR
             THEN r.positionOrder END)                  AS mean_finish_last_year,
    -- statusId 1 = "Finished" (same DNF definition as RelBench's driver-dnf task)
    AVG(CASE WHEN r.statusId != 1 THEN 1.0 ELSE 0 END)  AS dnf_rate,
    DATE_DIFF('day', MAX(r.date), t.date)               AS days_since_last_race,
    DATE_DIFF('day', ANY_VALUE(d.dob), t.date) / 365.25 AS age
FROM task_rows t
JOIN drivers d ON d.driverId = t.driverId
LEFT JOIN results r
       ON r.driverId = t.driverId
      AND r.date < t.date            -- only the past!
GROUP BY t.driverId, t.date
"""

def make_features(task_df):
    con = duckdb.connect()
    con.register("task_rows", task_df[["driverId", "date"]])
    con.register("drivers", drivers)
    con.register("results", results)
    feats = con.sql(FEATURE_SQL).df()
    con.close()
    # keep the task table's row order
    return task_df[["driverId", "date"]].merge(feats, on=["driverId", "date"], how="left")

X_train_full = make_features(train_df)
X_val_full = make_features(val_df)
FEATURES = [c for c in X_train_full.columns if c not in ("driverId", "date")]
X_train_full.head()

**Checkpoint:** why `r.date < t.date` and not just "all results of the driver"?
What would happen to the validation score if we removed that condition?

## 5. Demo: TabPFN

TabPFN is a transformer **pre-trained on millions of synthetic tables**. `fit` stores the training rows; `predict` runs one forward pass with the training rows as context ("in-context learning").

* GPU recommended. On CPU, keep ≤ ~5,000 training rows.
* The model weights download on first use and may require accepting a license on Hugging Face. **Do this before class.**

In [ ]:
!pip install -q tabpfn

In [ ]:
import torch
from tabpfn import TabPFNRegressor

MAX_TRAIN = 10_000 if torch.cuda.is_available() else 5_000
train_sub = (X_train_full.assign(y=train_df["position"].to_numpy())
             .sort_values("date").tail(MAX_TRAIN))      # most recent rows
print("training rows used:", len(train_sub), "| device:", "cuda" if torch.cuda.is_available() else "cpu")

reg = TabPFNRegressor()
reg.fit(train_sub[FEATURES], train_sub["y"])
pred_tabpfn = reg.predict(X_val_full[FEATURES])
report("TabPFN (flat features)", pred_tabpfn)

**Ablation:** is the gain from the model, or from the features?
Give TabPFN only a single feature, `mean_finish` (≈ the historical mean), and compare.

In [ ]:
reg_1f = TabPFNRegressor()
reg_1f.fit(train_sub[["mean_finish"]], train_sub["y"])
report("TabPFN (mean_finish only)", reg_1f.predict(X_val_full[["mean_finish"]]))

## 6. Compare

In [ ]:
pd.DataFrame(results_table).T.sort_values("val MAE")

Discussion
1. Does TabPFN beat the per-driver mean? By how much?
2. Is the improvement due to the model or to the features we gave it?
3. Which other tables (`qualifying`, `standings`, `constructors`, ...) could add signal, and how would you keep them leak-free?